In [89]:
import joblib
import pandas as pd
import os
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_predict

In [90]:
TRAINING_DATA_PATH = "./training_data/"
MODEL_OUTPUT_PATH = "./pkl/"

In [91]:
df = pd.read_csv(os.path.join(TRAINING_DATA_PATH, "training_data.csv"))

In [92]:
initial_count = df.shape[0]
df = df.drop_duplicates(subset=["query"])
print(f"Removed {initial_count - df.shape[0]} duplicate queries from training data.")

Removed 34 duplicate queries from training data.


In [93]:
X = df["query"]
y = df["label"]

In [94]:
model_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1,2))),
    ("nb", MultinomialNB(alpha=0.1))
])

In [95]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model_pipeline, X, y, cv=cv, scoring="accuracy")

print(f"\n--- K-Fold Cross Validation Results ---")
print(f"Scores for each of the 5 folds: {cv_scores}")
print(f"Average Accuracy: {cv_scores.mean():.4f}")
print(f"Standard Deviation: {cv_scores.std():.4f}\n")


--- K-Fold Cross Validation Results ---
Scores for each of the 5 folds: [0.921875 0.875    0.84375  0.953125 0.984375]
Average Accuracy: 0.9156
Standard Deviation: 0.0510



In [96]:
cv_predictions = cross_val_predict(model_pipeline, X, y, cv=cv)

results_df = pd.DataFrame({
    "query": X,
    "actual_label": y,
    "predicted_label": cv_predictions
})
errors_df = results_df[results_df["actual_label"] != results_df["predicted_label"]]

print(f"Total mistakes made across all 5 folds: {len(errors_df)}\n")
print(errors_df.to_string(index=False))

Total mistakes made across all 5 folds: 27

                                 query actual_label predicted_label
                          who is there         chat          search
                         make me laugh         chat          search
  find five cheap dishes i can prepare       search            chat
               play top hits in Poland      spotify          search
           distance from earth to mars       search         spotify
       what time does lulu lemon close       search            chat
            who wrote the great gatsby       search            chat
          top trending songs on tiktok       search         spotify
                 listen to the beatles      spotify          search
    play the soundtrack from gladiator      spotify          search
what is the weather like where you are         chat          search
      what is the best thing about you         chat          search
             find me a reason to smile         chat          search
  pl

In [97]:
model_pipeline.fit(X, y)

vectorizer = model_pipeline.named_steps["tfidf"]
nb_model = model_pipeline.named_steps["nb"]

vec_path = os.path.join(MODEL_OUTPUT_PATH, "vectorizer.pkl")
nb_path = os.path.join(MODEL_OUTPUT_PATH, "multinomial_naive_bayes.pkl")

joblib.dump(vectorizer, vec_path)
joblib.dump(nb_model, nb_path)

['./pkl/multinomial_naive_bayes.pkl']